## Week 2 Day 3

Now we get to more detail:

1. Different models

2. Structured Outputs

3. Guardrails

In [1]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from pydantic import BaseModel

In [2]:
load_dotenv(override=True)

True

In [3]:
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
if openrouter_api_key:
    print(f"OpenAI API Key exists and begins {openrouter_api_key[:9]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-or-v1-


In [4]:
"""openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")"""

'openai_api_key = os.getenv(\'OPENAI_API_KEY\')\ngoogle_api_key = os.getenv(\'GOOGLE_API_KEY\')\ndeepseek_api_key = os.getenv(\'DEEPSEEK_API_KEY\')\ngroq_api_key = os.getenv(\'GROQ_API_KEY\')\n\nif openai_api_key:\n    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")\nelse:\n    print("OpenAI API Key not set")\n\nif google_api_key:\n    print(f"Google API Key exists and begins {google_api_key[:2]}")\nelse:\n    print("Google API Key not set (and this is optional)")\n\nif deepseek_api_key:\n    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")\nelse:\n    print("DeepSeek API Key not set (and this is optional)")\n\nif groq_api_key:\n    print(f"Groq API Key exists and begins {groq_api_key[:4]}")\nelse:\n    print("Groq API Key not set (and this is optional)")'

In [5]:
professional_sales_agent_instructions = "You are a sales agent working for ComplAI, a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. You write professional, serious cold emails." # Professional Sales Agent

humorous_sales_agent_instructions = "You are a humorous, engaging sales agent working for ComplAI, a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. You write witty, engaging cold emails that are likely to get a response."

concise_sales_agent_instructions = "You are a busy sales agent working for ComplAI, a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. You write concise, to the point cold emails."

### It's easy to use any models with OpenAI compatible endpoints

In [6]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# OpenRouter IDs contain "/"; a plain model string is parsed as "provider/model" by the SDK — use OpenAIChatCompletionsModel + this client.
# The free Llama 3.3 70B endpoint is often 429 upstream (e.g. Venice); Nemotron is a separate free route with tool support.
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
nemotron_model = OpenAIChatCompletionsModel(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    openai_client=openrouter_client,
)

### Creation of the Email Sender Agent

This email Sender Agent frames the subject to the email message, converts the email message to HTML markdown and sends the email

In [ ]:
subject_instructions = "You can write a subject for a cold sales email. You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. You are given a text email body which might have some markdown and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model=nemotron_model)
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model=nemotron_model)
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")

In [8]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("amithvardhangd3@gmail.com")  # Change to your verified sender
    to_email = To("amithvardhangd3@gmail.com")  # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [9]:
email_tools = [subject_tool, html_tool, send_html_email]

In [10]:
from typing import Any


instructions ="You are an email formatter and sender. You receive the body of an email to be sent. You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. Finally, you use the send_html_email tool to send the email with the subject and HTML body."

# The following agent is to frame the subject to the email message, convert the email message to HTML markdown and send the email
emailer_agent = Agent[Any](
    name="Email Manager",
    instructions=instructions,
    tools=email_tools,
    model=nemotron_model,
    handoff_description="Convert an email to HTML and send it")

### Creating the main agent for the entire process

In [11]:
qwen_model = OpenAIChatCompletionsModel(model="qwen/qwen3.6-plus:free", openai_client=openrouter_client)
step_model = OpenAIChatCompletionsModel(model="stepfun/step-3.5-flash:free", openai_client=openrouter_client)
nemotron_model = OpenAIChatCompletionsModel(model="nvidia/nemotron-3-nano-30b-a3b:free", openai_client=openrouter_client)

In [12]:
qwen_professional_agent = Agent(name="Professional Sales Agent using Qwen", instructions=professional_sales_agent_instructions, model=qwen_model)
step_humorous_agent =  Agent(name="Humorous Sales Agent using Step", instructions=humorous_sales_agent_instructions, model=step_model)
nemotron_concise_agent = Agent(name="Concise Sales Agent using Nemotron", instructions=concise_sales_agent_instructions, model=nemotron_model)

In [13]:
description = "Write a cold sales email"

qwen_professional_tool = qwen_professional_agent.as_tool(tool_name="qwen_professional_tool", tool_description=description)
step_humorous_tool = step_humorous_agent.as_tool(tool_name="step_humorous_tool", tool_description=description)
nemotron_concise_tool = nemotron_concise_agent.as_tool(tool_name="nemotron_concise_tool", tool_description=description)

In [14]:
tools = [qwen_professional_tool, step_humorous_tool, nemotron_concise_tool]
handoffs = [emailer_agent]

In [15]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""

# The following is the main agent that uses the three email generator agent tools and hands off the control to the agent that sends emails
sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model=nemotron_model)

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is no

In [18]:
print(result.final_output)

✅ The cold‑sales email has been sent successfully!  

**Subject used:** “Your Decisions, Powered by AI (No Extra Staff Needed)”  
**Recipient:** Dear CEO (from Alice)  
**Status:** Email dispatched via the email manager.

*Note: Be sure to replace the placeholders (e.g., [Calendar Link], [Phone], [Email], [Website]) with the actual details before future sends.*


## Check out the trace:

https://platform.openai.com/traces

In [ ]:
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str

guardrail_agent = Agent( 
    name="Name check",
    instructions="Check if the user is including someone's personal name in what they want you to do.",
    output_type=NameCheckOutput,
    model=nemotron_model
)

In [39]:
@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info={"found_name": result.final_output},tripwire_triggered=is_name_in_message)

In [40]:
careful_sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=[emailer_agent],
    model=nemotron_model,
    input_guardrails=[guardrail_against_name]
    )

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)

In [41]:
print(result.final_output)

✅ The cold‑sales email has been sent successfully!  

- **Subject:** Boost CEO productivity& cut communication costs with ComplAI 🚀  
- **Recipient:** Dear CEO (addressed from Alice)  
- **Content:** HTML email highlighting ComplAI’s AI‑driven platform, productivity boost, cost savings, and compliance benefits, with a CTA to schedule a 15‑minute demo.  

The message was delivered to the sales prospects list via the email manager. Let me know if you’d like any follow‑up or additional variations!


## Check out the trace:

https://platform.openai.com/traces

In [37]:

message = "Send out a cold sales email addressed to Dear CEO from Head of Business Development"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">• Try different models<br/>• Add more input and output guardrails<br/>• Use structured outputs for the email generation
            </span>
        </td>
    </tr>
</table>